In [21]:
import os
import sys
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from groq import Groq
from langchain_core.tools import tool
import pandas as pd
from typing import TypedDict
from langgraph.graph import StateGraph
from langchain_experimental.agents.agent_toolkits.pandas.base import create_pandas_dataframe_agent
import pandas as pd

load_dotenv()


ImportError: cannot import name 'format_tool_to_openai_function' from 'langchain_core.utils.function_calling' (d:\Ascent_projects\Hack_etl\venv\Lib\site-packages\langchain_core\utils\function_calling.py)

In [ ]:
api_key = os.getenv("GROQ_API_KEY")

In [ ]:
def modify_df_node(state: WorkflowState) -> WorkflowState:
    df = pd.read_csv(state["csv_path"])
    llm = ChatGroq(
        temperature=0,
        model_name="llama-3-70b-versatile",
        groq_api_key=os.getenv("GROQ_API_KEY")
    )
    agent = create_pandas_dataframe_agent(llm, df, verbose=True)
    result = agent.run(state["user_input"])
    print("Groq agent response:", result)
    state["modification_result"] = result
    return state


In [ ]:
def save_csv_node(state: WorkflowState) -> WorkflowState:
    df = pd.read_csv(state["csv_path"])
    df.to_csv(state["result_path"], index=False)
    print(f"Saved modified CSV to {state['result_path']}")
    return state

In [ ]:
from langgraph.graph import StateGraph

builder = StateGraph(WorkflowState)

builder.add_node("load_csv", load_csv_node)
builder.add_node("modify_df", modify_df_node)
builder.add_node("save_csv", save_csv_node)

builder.set_entry_point("load_csv")
builder.add_edge("load_csv", "modify_df")
builder.add_edge("modify_df", "save_csv")

graph = builder.compile()

In [ ]:
import json
config_path = "../Config/config.json"
with open(config_path) as config_file:
    config = json.load(config_file)
    print("Configuration loaded:", config)

Configuration loaded: {'source_input_path': 'Data/source/synthetic_data.csv', 'transformed_output_path': 'Data/Transformation/transformed_data.csv', 'destination_output_path': 'Data/Destination/final_data.csv'}


In [ ]:
initial_state = {
    "csv_path": config["source_input_path"],
    "user_input": "check for null values and remove any rows that contain them",
    "result_path": config["destination_output_path"]
}

final_state = graph.invoke(initial_state)

FileNotFoundError: [Errno 2] No such file or directory: 'Data/source/synthetic_data.csv'

In [22]:
12+12+12+6+3

45